# Content Platforms Comparison — Netflix vs Spotify

This notebook generates the single side-by-side chart used in the README:
**Netflix's top genres by title count** next to **Spotify's top genres by average popularity**.

These two rankings look structurally similar — bar charts of genres — but they measure
fundamentally different things. Netflix's ranking reflects **acquisition choices** (what the
platform decided to add). Spotify's reflects **audience behavior** (what listeners actually
streamed). The visual juxtaposition is the comparison's central insight.

Data is pulled directly from the two analysis repos via raw GitHub URLs — no local files
needed, and the chart stays reproducible by anyone.


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Raw URLs to the data files in the two analysis repos
NETFLIX_URL = "https://raw.githubusercontent.com/Mallika56/netflix-analytics/main/data/netflix_titles.csv"
SPOTIFY_URL = "https://raw.githubusercontent.com/Mallika56/spotify-analytics/main/data/spotify_tracks.csv"

print("URLs configured.")

## Load both datasets

In [ ]:
netflix = pd.read_csv(NETFLIX_URL)
spotify = pd.read_csv(SPOTIFY_URL)

print(f"Netflix: {netflix.shape[0]:,} titles, {netflix.shape[1]} columns")
print(f"Spotify: {spotify.shape[0]:,} tracks, {spotify.shape[1]} columns")

## Compute the two rankings

**Netflix:** explode the multi-genre `listed_in` field, count titles per genre, take top 10.
This measures *catalog volume* — how much of each genre Netflix chose to license or produce.

**Spotify:** group by genre, average popularity, take top 10.
This measures *audience engagement* — which genres listeners actually keep streaming.

In [ ]:
# Netflix — top genres by title count
netflix_genres = (
    netflix.assign(genre=netflix["listed_in"].str.split(", "))
           .explode("genre")
           .groupby("genre")
           .size()
           .sort_values(ascending=False)
           .head(10)
           .reset_index(name="titles")
)
print("Netflix top genres (by count):")
print(netflix_genres)

In [ ]:
# Spotify — top genres by average popularity
spotify_genres = (
    spotify.dropna(subset=["popularity", "genre"])
           .groupby("genre")["popularity"]
           .mean()
           .round(1)
           .sort_values(ascending=False)
           .head(10)
           .reset_index(name="avg_popularity")
)
print("Spotify top genres (by avg popularity):")
print(spotify_genres)

## The side-by-side chart

Two horizontal bar charts, sharing the same visual treatment so the eye reads them as
parallel — but with different x-axes (titles vs avg popularity) because they measure
different things. The shared structure carries the visual punch; the different axes carry
the analytical point.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Netflix panel
axes[0].barh(netflix_genres["genre"][::-1], netflix_genres["titles"][::-1], color="#E50914")
axes[0].set_title("Netflix — Top Genres by Title Count\n(what the platform acquired)",
                  fontsize=12, pad=12)
axes[0].set_xlabel("Number of titles")
axes[0].spines[["top", "right"]].set_visible(False)

# Spotify panel
axes[1].barh(spotify_genres["genre"][::-1], spotify_genres["avg_popularity"][::-1],
             color="#1DB954")
axes[1].set_title("Spotify — Top Genres by Avg Popularity\n(what audiences streamed)",
                  fontsize=12, pad=12)
axes[1].set_xlabel("Average popularity (0-100)")
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Two Platforms, Two Definitions of 'Popular'", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../visuals/top_genres_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../visuals/top_genres_comparison.png")

## What this chart shows (and what it doesn't)

The two panels look like the same chart — bar rankings of genres — but the *units differ*.
Netflix's bars count *titles*. Spotify's bars average *audience attention scores*.
That asymmetry is the point: the same word "popular" carries different meaning on each
platform, because the platforms expose different sides of the content lifecycle.

The chart does **not** compare absolute scales (you can't say "Spotify pop > Netflix
International Movies"). It compares the *shape* of each ranking and the *strategic logic*
each implies — which is the only honest comparison the data supports.